# Day 9: Data quality checks for NFL data

**Phase 1: Foundations & Infrastructure**

## Objective
Data quality checks for NFL data

## Task
- Executed high-level data extraction and ingestion for NFL play-by-play data (2025 season) using `nfl_data_py`.
- Verified Primary Key integrity (duplicate check on `game_id` + `play_id`) to ensure row uniqueness.
- Validated logical consistency against the football rulebook (audited impossible downs and out-of-bounds 'yards to go').
- Audited Expected Points Added (EPA) distribution consistency and verified the absence of extreme statistical outliers.
- Implemented a three-tier advanced validation and cleaning strategy:
    - **Strategy 1:** Differentiated between strategic nulls and actual data errors (e.g., auditing missing `air_yards` specifically on pass plays).
    - **Strategy 2:** Validated identifier referential integrity (ensured all active pass plays map to a registered `passer_player_id`).
    - **Strategy 3:** Resolved "Penalty Chaos" by filtering out ghost plays (nullified by accepted penalties) to isolate a clean dataset for player efficiency metrics.
- Generated an automated, professional HTML profiling report using `ydata-profiling` saved in `reports/html/`.
**Deliverable:** Notebook `day009_nfl_data_quality.ipynb` + HTML profiling report.

---

### Instructions:
*Treat this as your course workbook. Write your code, notes, or execution steps below.*

Remember: 
- Document your decisions.
- Use clear variable names.
- If today's task involves creating a script in `src/`, a dashboard in `apps/`, or documentation in `docs/`, you can use this notebook to test your logic or simply write your reflections and proofs of execution (e.g. running terminal commands with `!`).



In [ ]:
pip install statsbombpy

In [ ]:
pip install nfl_data_py

In [ ]:
# Downgrade Pandas to the last stable 1.x version to ensure compatibility
%pip install "pandas<2.0.0"

In [1]:
import sys
import os
import pandas as pd

# 1. Add the project root path to the system so Python finds 'src'
# We go up two levels: from phase1_foundations -> notebooks -> root
sys.path.append(os.path.abspath("../../"))

# 2. Now we import the correct function for the NFL
from src.data.loaders import load_nfl

# Define the years we want to analyze
years_to_download = [2025]
print(f"Starting NFL Play-by-Play download for season(s): {years_to_download}...")
print("This might take a couple of minutes depending on your connection...")

# Use our unified function from Day 7 (now with load_nfl!)
pbp_df = load_nfl(years=years_to_download, data_type="pbp")

# Validate that the download was successful
if pbp_df is not None:
    print("\nDownload and ingestion successful!")
    print(f"Raw dataset dimensions: {pbp_df.shape[0]} rows (plays) x {pbp_df.shape[1]} columns")
    
    # Take a quick look at the first 3 plays
    display(pbp_df[['game_id', 'play_id', 'home_team', 'away_team', 'desc']].head(3))
else:
    print("\n Download error. Check your loaders.py file or your connection.")

2026-04-30 21:08:02 - INFO - Local cache found. Loading Parquet data from: C:\Users\david\Desktop\David\Documentos\David\GitHub\100-days-Data-Sports-Challenge\data\raw/nfl/nfl_pbp_2025.parquet


Starting NFL Play-by-Play download for season(s): [2025]...
This might take a couple of minutes depending on your connection...

Download and ingestion successful!
Raw dataset dimensions: 48771 rows (plays) x 397 columns


,game_id,play_id,home_team,away_team,desc
0,2025_01_ARI_NO,1.0,NO,ARI,GAME
1,2025_01_ARI_NO,40.0,NO,ARI,19-B.Grupe kicks 65 yards from NO 35 to ARI 0....
2,2025_01_ARI_NO,63.0,NO,ARI,(14:56) 6-J.Conner right tackle to ARI 25 for ...


In [2]:
print("--- Point 1: Primary Key Integrity Check ---")

# 1. Identify the composite primary key columns
pk_columns = ['game_id', 'play_id']

# 2. Count how many rows are exact duplicates based on these two columns
duplicate_plays = pbp_df.duplicated(subset=pk_columns).sum()

print(f"Duplicate plays found (Game ID + Play ID): {duplicate_plays}")

# 3. Investigate if duplicates exist
if duplicate_plays > 0:
    print("WARNING: Duplicates detected! Showing the first few:")
    # keep=False shows all copies of the duplicated rows
    duplicates_df = pbp_df[pbp_df.duplicated(subset=pk_columns, keep=False)]
    display(duplicates_df[pk_columns + ['desc', 'play_type']].head())
else:
    print("Primary Key integrity is perfect. No duplicates found.")

--- Point 1: Primary Key Integrity Check ---
Duplicate plays found (Game ID + Play ID): 0
Primary Key integrity is perfect. No duplicates found.


In [3]:
print("--- Point 2: Logical Consistency Check (Down & Distance) ---")

# 1. Filter for impossible downs (strictly greater than 4)
# Note: Kickoffs and Extra Points often have NaN (null) downs, which is expected.
impossible_downs = pbp_df[pbp_df['down'] > 4]

# 2. Filter for impossible yards to go (negative or mathematically out of bounds)
impossible_yards = pbp_df[(pbp_df['ydstogo'] < 0) | (pbp_df['ydstogo'] > 100)]

print(f"Plays with a 5th down (or more): {len(impossible_downs)}")
print(f"Plays with invalid 'yards to go': {len(impossible_yards)}")

# 3. Display anomalies if they exist
if len(impossible_downs) > 0 or len(impossible_yards) > 0:
    print("\nWARNING: Logical anomalies detected in the rulebook! Inspecting...")
    anomaly_cols = ['game_id', 'play_id', 'down', 'ydstogo', 'play_type', 'desc']
    
    if len(impossible_downs) > 0:
        print("-> Showing impossible downs:")
        display(impossible_downs[anomaly_cols].head())
        
    if len(impossible_yards) > 0:
        print("-> Showing invalid yards to go:")
        display(impossible_yards[anomaly_cols].head())
else:
    print("\nDown and distance logic is perfectly bounded.")

--- Point 2: Logical Consistency Check (Down & Distance) ---
Plays with a 5th down (or more): 0
Plays with invalid 'yards to go': 0

Down and distance logic is perfectly bounded.


In [4]:
print("--- Point 3: EPA Distribution Check ---")

# 1. Get the statistical summary of the EPA column
epa_summary = pbp_df['epa'].describe()
print("EPA Statistical Summary:")
print(epa_summary)

# 2. Check for extreme outliers (EPA > 20 or EPA < -20)
extreme_epa = pbp_df[(pbp_df['epa'] > 20) | (pbp_df['epa'] < -20)]

print(f"\nPlays with extreme EPA (>10 or <-10): {len(extreme_epa)}")

if len(extreme_epa) > 0:
    print("WARNING: Extreme EPA values detected. Inspecting top plays:")
    # Sort by absolute EPA to see the most extreme cases first
    extreme_sorted = extreme_epa.reindex(extreme_epa['epa'].abs().sort_values(ascending=False).index)
    display(extreme_sorted[['game_id', 'play_id', 'play_type', 'desc', 'epa']].head(3))
else:
    print("EPA values are within a realistic football range.")

--- Point 3: EPA Distribution Check ---
EPA Statistical Summary:
count    48201.000000
mean         0.013139
std          1.258956
min        -12.658112
25%         -0.551978
50%          0.000000
75%          0.551108
max          7.967405
Name: epa, dtype: float64

Plays with extreme EPA (>10 or <-10): 0
EPA values are within a realistic football range.


In [5]:
print("--- Point 4: Strategic Nulls vs. Data Errors ---")

# 1. We want to analyze 'air_yards' (how far the ball traveled in the air)
# We group by 'play_type' and calculate the percentage of missing values (NaNs)
missing_air_yards = pbp_df.groupby('play_type')['air_yards'].apply(lambda x: x.isnull().mean() * 100).round(2)

print("Percentage of missing 'air_yards' by play type:")
# Filter only the most common play types for a clean view
common_plays = ['pass', 'run', 'punt', 'kickoff']
display(missing_air_yards.loc[missing_air_yards.index.isin(common_plays)])

# 2. Check for ACTUAL errors: Passes that are missing air_yards
# A pass should technically have an air_yards value, even if it's 0 (screen pass).
# However, sometimes penalties or sacks mess this up.
error_passes = pbp_df[(pbp_df['play_type'] == 'pass') & (pbp_df['air_yards'].isnull())]

print(f"\nPass plays with missing 'air_yards': {len(error_passes)}")
if len(error_passes) > 0:
    print("WARNING: Some pass plays are missing air yard data. Inspecting anomalies:")
    display(error_passes[['game_id', 'play_id', 'desc', 'air_yards', 'sack', 'penalty']].head(5))
else:
    print("All pass plays have valid air_yards data.")

--- Point 4: Strategic Nulls vs. Data Errors ---
Percentage of missing 'air_yards' by play type:


play_type
kickoff    100.00
pass         7.33
punt       100.00
run        100.00
Name: air_yards, dtype: float64


Pass plays with missing 'air_yards': 1447


,game_id,play_id,desc,air_yards,sack,penalty
4,2025_01_ARI_NO,115.0,(13:40) 1-K.Murray sacked at ARI 25 for -11 ya...,NaN,1.0,0.0
15,2025_01_ARI_NO,408.0,(8:55) (No Huddle) 1-K.Murray sacked ob at NO ...,NaN,1.0,0.0
18,2025_01_ARI_NO,486.0,(8:05) (Shotgun) 1-K.Murray sacked at NO 39 fo...,NaN,1.0,0.0
87,2025_01_ARI_NO,2303.0,"(12:55) (No Huddle, Shotgun) 2-S.Rattler sacke...",NaN,1.0,0.0
116,2025_01_ARI_NO,3024.0,(1:40) (Shotgun) 1-K.Murray sacked at ARI 39 f...,NaN,1.0,0.0


In [6]:
print("--- Point 5: Identifier Validation (Rosters) ---")

# 1. Look for pass plays (excluding sacks) where the passer ID is missing
# We exclude sacks because we already know the ball wasn't thrown
ghost_passers = pbp_df[
    (pbp_df['play_type'] == 'pass') & 
    (pbp_df['sack'] == 0) & 
    (pbp_df['passer_player_id'].isna())
]

print(f"Pass plays missing a passer ID: {len(ghost_passers)}")

# 2. Inspect these anomalies if they exist
if len(ghost_passers) > 0:
    print("\nWARNING: Found passes without a registered Quarterback ID. Inspecting:")
    # We display 'play_type_nfl' and the description to understand the context
    display(ghost_passers[['game_id', 'play_id', 'desc', 'play_type_nfl']].head(5))
else:
    print("\nAll active pass plays are correctly linked to a player ID.")

--- Point 5: Identifier Validation (Rosters) ---
Pass plays missing a passer ID: 0

All active pass plays are correctly linked to a player ID.


In [7]:
print("--- Advanced Challenge: The Penalty Chaos ---")

# 1. Check how many plays had a penalty flag
total_penalties = pbp_df[pbp_df['penalty'] == 1].shape[0]
print(f"Total plays with a penalty flag: {total_penalties}")

# 2. The Golden Rule of NFL Data: 
# Accepted penalties that nullify a play usually change the 'play_type' to 'no_play'.
# If we are calculating actual rushing/passing stats, we MUST drop them.
# Note: Declined penalties are kept because the play physically counted.

efficiency_df = pbp_df[pbp_df['play_type'] != 'no_play'].copy()

# 3. Let's see the impact of our filter
original_rows = pbp_df.shape[0]
clean_rows = efficiency_df.shape[0]
ghost_plays_removed = original_rows - clean_rows

print(f"\nOriginal dataset rows: {original_rows}")
print(f"Clean dataset rows (for player stats): {clean_rows}")
print(f"Ghost plays dropped (Nullified by penalties): {ghost_plays_removed}")

--- Advanced Challenge: The Penalty Chaos ---
Total plays with a penalty flag: 3559

Original dataset rows: 48771
Clean dataset rows (for player stats): 44039
Ghost plays dropped (Nullified by penalties): 4732


In [ ]:
pip install ydata_profiling

In [ ]:
%pip install "pandas<2.1.0" "ydata-profiling" --upgrade


In [10]:
from ydata_profiling import ProfileReport

print("--- Generating Professional Data Quality Report (NFL Day 9) ---")

# 1. Select key columns to avoid memory issues and focus on what matters
# We pick a mix of identifiers, situational data, and performance metrics
core_columns = [
    'play_id', 'game_id', 'drive', 'qtr', 'down', 'ydstogo', 'yardline_100',
    'play_type', 'yards_gained', 'air_yards', 'yards_after_catch',
    'epa', 'wpa', 'cp', 'cpoe', 'success', 
    'passer_player_name', 'rusher_player_name', 'receiver_player_name',
    'penalty', 'qb_hit', 'sack', 'touchdown'
]

# Filter our clean efficiency dataframe
report_df = efficiency_df[core_columns].copy()

# 2. Initialize the Profile Report
profile = ProfileReport(
    report_df, 
    title="NFL Play-by-Play Data Quality Audit - 2025 Season",
    explorative=True,
    minimal=True # Recommended for large NFL datasets to stay fast
)

# 3. Save to your professional reports folder
report_path = "../../reports/html/nfl_pbp_quality_report.html"
profile.to_file(report_path)

print(f"Quality report successfully saved to: {report_path}")

c:\Users\david\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-30 21:08:05 - INFO - Pandas backend loaded 2.0.3
2026-04-30 21:08:05 - INFO - Numpy backend loaded 1.26.4
2026-04-30 21:08:05 - INFO - Pyspark backend NOT loaded
2026-04-30 21:08:05 - INFO - Python backend loaded
C:\Users\david\AppData\Local\Temp\ipykernel_29068\3567369654.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


--- Generating Professional Data Quality Report (NFL Day 9) ---


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 242.64it/s]

Quality report successfully saved to: ../../reports/html/nfl_pbp_quality_report.html
